# Parquet WellViz DataFrame Inspection

This notebook inspects the indexed SCREEN WellViz Parquet package with tabular previews, profiling summaries, and visual quality-control plots.

## 1. Import Libraries and Configure Display

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.precision", 4)
sns.set_theme(style="whitegrid")

print("Plotly available:", PLOTLY_AVAILABLE)

## 2. Load one selected scenario

Set `CASE_NAME` in the next cell to the `case_name` you want to inspect. The
notebook loads only that case's indexed Parquet package and displays its
`scenario.json` provenance. Build a package for each batch case before
comparing them, for example:

```bash
uv run python runscripts/export_wellviz_indexed.py \
    --results-root work/results \
    --case baseline \
    --output-dir work/results/baseline_wellviz_indexed \
    --all-records
```

Repeat the export with the other case names and change `CASE_NAME` to inspect
them. `INIT` properties describe the static grid and may be identical between
pressure scenarios. Select `UNRST` and a later record for dynamic pressure or
saturation differences.

In [ ]:
import json

RESULTS_ROOT = Path("work/results")
CASE_NAME = "baseline"  # Change this to another case_name from the batch run.
PACKAGE_DIR = RESULTS_ROOT / f"{CASE_NAME}_wellviz_indexed"
PARQUET_PATH = PACKAGE_DIR / "data.parquet"
SCENARIO_PATH = RESULTS_ROOT / CASE_NAME / "scenario.json"

if not PARQUET_PATH.exists():
    raise FileNotFoundError(
        f"No indexed WellViz package found for {CASE_NAME!r}: {PARQUET_PATH}. "
        "Run export_wellviz_indexed.py for this case first."
    )

df = pd.read_parquet(PARQUET_PATH)
source_frames = {
    source: frame.reset_index(drop=True)
    for source, frame in df.groupby("source", sort=True)
}

if SCENARIO_PATH.exists():
    scenario = json.loads(SCENARIO_PATH.read_text(encoding="utf-8"))
    scenario_frame = pd.DataFrame([scenario]).T.rename(columns={0: "value"})
    display(scenario_frame)
else:
    scenario = None
    print(f"No provenance file found at {SCENARIO_PATH}")

print(f"Loaded case {CASE_NAME!r}: {len(df):,} rows from {PARQUET_PATH}")
df.head()

## 3. Quick Structural Inspection

## 3. Compare scenario values

Use the same source, property, timestep, and J column for each case. A
comparison of `INIT` is expected to show the same static geometry and initial
properties in many cases. Compare `UNRST` records, especially later records,
to see pressure-driven differences.

In [ ]:
CASE_NAMES = ["baseline", "pressure_minus_25", "pressure_minus_50", "pressure_minus_75"]
SOURCE = "UNRST"
PROPERTY = "PRESSURE"
RECORD = 1
J_COLUMN = 0

comparison_frames = []
for case_name in CASE_NAMES:
    case_path = RESULTS_ROOT / f"{case_name}_wellviz_indexed" / "data.parquet"
    if not case_path.exists():
        print(f"Skipping {case_name!r}: {case_path} does not exist")
        continue
    case_df = pd.read_parquet(case_path)
    selected = case_df[
        (case_df["source"] == SOURCE)
        & (case_df["property"] == PROPERTY)
        & (case_df["record"] == RECORD)
        & (case_df["j_column"] == J_COLUMN)
    ].copy()
    selected["case_name"] = case_name
    comparison_frames.append(selected)

if not comparison_frames:
    raise FileNotFoundError("No selected scenario packages were found for comparison.")

comparison = pd.concat(comparison_frames, ignore_index=True)
summary = comparison.groupby("case_name")["value"].agg(["count", "min", "max", "mean"])
display(summary)

if PLOTLY_AVAILABLE:
    fig = px.line(
        comparison.sort_values(["case_name", "z", "x"]),
        x="x",
        y="value",
        color="case_name",
        facet_col="j_column",
        title=f"{SOURCE} {PROPERTY}, record {RECORD}, J column {J_COLUMN}",
    )
    fig.show()
else:
    comparison.boxplot(column="value", by="case_name", rot=45)
    plt.suptitle("")
    plt.title(f"{SOURCE} {PROPERTY}, record {RECORD}")
    plt.show()